In [1]:
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 12.3 MB/s eta 0:00:00


In [2]:
import faiss
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

In [3]:
from google.colab import files

uploaded = files.upload()

Saving d_t4_2_2_technical_documentation_the_for_web_based_tool_final_version.pdf to d_t4_2_2_technical_documentation_the_for_web_based_tool_final_version.pdf


In [4]:
pdf_file = list(uploaded.keys())[0]

reader = PdfReader(pdf_file)

document_text = ""

for page in reader.pages:
    text = page.extract_text()

    if text:
        document_text += text + "\n"

print("Characters Loaded:", len(document_text))

Characters Loaded: 23639


In [5]:
def create_chunks(
    text,
    chunk_size=700,
    overlap=150
):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [6]:
chunks = create_chunks(document_text)

print("Total Chunks:", len(chunks))

Total Chunks: 43


In [7]:
print("Loading Embedding Model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

Loading Embedding Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [8]:
print("Generating Embeddings...")

chunk_embeddings = embedding_model.encode(
    chunks,
    batch_size=8,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Shape:", chunk_embeddings.shape)

Generating Embeddings...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Shape: (43, 384)


In [9]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(chunk_embeddings)

print(
    "Vectors Stored:",
    index.ntotal
)

Vectors Stored: 43


In [10]:
def semantic_search(
    query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved_chunks = []

    for idx in indices[0]:

        retrieved_chunks.append(
            chunks[idx]
        )

    return retrieved_chunks

In [11]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("FLAN-T5 Loaded")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 Loaded


In [12]:
def generate_answer(
    question,
    context
):

    prompt = f"""
Answer the question using the context.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [13]:
def ask_question(
    question
):

    retrieved_chunks = semantic_search(
        question
    )

    context = "\n".join(
        retrieved_chunks
    )

    answer = generate_answer(
        question,
        context
    )

    return answer

In [14]:
question = "What is the purpose of the web based tool?"

answer = ask_question(
    question
)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What is the purpose of the web based tool?

Answer:
to operate in real-time with no expected extra-traffic time periods


In [15]:
questions = [

    "What is the purpose of the web based tool?",

    "What are the system characteristics?",

    "What are the infrastructure services?",

    "What are the layers of system architecture?",

    "What is project administration?",

    "What is project grid?"
]

In [ ]:
for q in questions:

    print("="*80)

    print("Question:")
    print(q)

    print()

    print(
        ask_question(q)
    )

    print()

Question:
What is the purpose of the web based tool?

to operate in real-time with no expected extra-traffic time periods

Question:
What are the system characteristics?

Systems characteristics are described upon the formal demands of the Project.

Question:
What are the infrastructure services?

#1 The Infrastructure shall allow only an authorized access to the system. #2 It shall provide the following functionalities: a. Security – strong user name/password policy b. Audit – auditing of user access and data access c. Logging – automatically creating change-log for all data Technical documentation for the web-based tool Page 7 Ver. 1.1 3 SYSTEM CONTEXT #1 The whole system is constructed as a web solution with different types of access: a. System administrator – inserts cruc ial information such as information about users b. Local administrator – can perform CRUD operations on single PPP projects c. Registered user – can access

Question:
What are the layers of system architecture?

